In [145]:
import sqlite3
import pandas as pd

conn = sqlite3.Connection('../backend/data.db')

df = pd.read_sql('''
    select vector, chunk
    from embeddings 
    join chunks using (chunk_id)
    order by random() 
    limit 200000
''', conn)

In [146]:
df

,vector,chunk
0,b'\n)\x9f.\xff\xa8T\x8dA+\xee\xa5\xaf\x11~)!\x...,honest; some review columns have such titles a...
1,b'\xef\x9c\x9e\'\x08\xa8\x9a)\xcb(\xdd\xa1b\xa...,European Grand Prix in Valencia.[201] Followin...
2,"b'\xb7\x1bb,\xbf*\\#\xd3\x9b\x86\x16\x07\xa8\x...",Ashkenazic Jews to date the Talmud from its be...
3,b'\x0e-\xfb\x1dy*k-\x07\x9aS*d\xac\xfb\xad\xec...,"Alabama, 1844–1994. Mobile, Alabama: Congregat..."
4,"b'\x9a\x0e\x8b$\x8d-""\x10\'\xa6\xee$\xf8\xa2\r...",more recently Celtic Nation F.C.). The second ...
...,...,...
199995,b'\x16+\xa9(\xc9*)*v\xad\xde\xa9\xbf\'~#\x04\x...,recommended that women in areas affected by th...
199996,"b'\xd6(u#\xa8\xa8""""\xeb%-\x99\xeb)\xd9\x9e6\xa...",brighter display by passing the electron beam ...
199997,b'+\xa0\x01\xa8\xe5*\x96\xa9\xb9-\x80\xab\xc7\...,"definitions, or else whose approach is a hybri..."
199998,b'\xd0+/\x11\xfa%\x8f-\x9b\x1bs(\x81%\xfe+\xac...,"with 40 family members present. His sons, the ..."


In [147]:
# pd.read_sql('select count(*) from embeddings', conn)

In [148]:
import base64
import numpy as np

In [149]:
df['vector'] = df['vector'].apply(
    lambda x: np.frombuffer(x, dtype=np.float16)
)

In [150]:
m = df['vector']
m = np.array([x for x in m])

In [151]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

X = m

# max_k = min(30, X.shape[0])  
# k_values = range(1, 1000, 100)

# inertias = []
# for k in k_values:
#     print(k)
#     km = KMeans(n_clusters=k, n_init="auto", random_state=42)
#     km.fit(X)
#     inertias.append(km.inertia_)  

# plt.figure()
# plt.plot(list(k_values), inertias, marker="o")
# plt.title("Elbow Method (Inertia vs k)")
# plt.xlabel("Number of clusters (k)")
# plt.ylabel("Inertia (within-cluster SSE)")
# plt.xticks(list(k_values))
# plt.grid(True)
# plt.show()

In [152]:
k = 1000
kmeans = KMeans(n_clusters=k, n_init="auto", random_state=42)
labels = kmeans.fit_predict(X)
centers = kmeans.cluster_centers_

In [153]:
df['label'] = labels

In [248]:
df.groupby('label').size().sort_values()[: 600]

label
572     56
798     63
165     73
968     75
237     78
      ... 
192    207
31     207
676    207
781    207
830    207
Length: 600, dtype: int64

In [249]:
chosen_label = 31

In [256]:
c = df[df.label == chosen_label].sample().iloc[0]['chunk']
print(c[: 100])
print(c[100 : 200])
print(c[200 :])

Treweek JB (December 2011). "Vaccines targeting drugs of abuse: is the glass half-empty or half-full
?". Nature Reviews. Immunology. 12 (1): 67–72. doi:10.1038/nri3130. PMID 22173478. ^ Anthenelli RM, 
Somoza E (September 2010). "Vaccine for cocaine addiction: A promising new immunotherapy" (PDF).


In [251]:
from scipy.spatial.distance import cosine

chosen_df = df[df.label == chosen_label].copy()
chosen_center = centers[chosen_label]

dists = []

for idx, row in chosen_df.iterrows():
    dists.append(cosine(row['vector'], chosen_center))

chosen_df['dist_from_center'] = dists
c = chosen_df.sort_values('dist_from_center').iloc[0]['chunk']
print(c[: 100])
print(c[100 : 200])
print(c[200 :])

PM (September 26, 2012). Evidence-Based Endocrinology. Lippincott Williams & Wilkins. pp. 217–. ISBN
 978-1-4511-7146-4. Archived from the original on January 11, 2023. Retrieved May 19, 2018. ^ a b St
einberger E, Ayala C, Hsi B, Smith KD, Rodriguez-Rigau LJ, Weidman


In [252]:
from collections import Counter
t = []
for x in chosen_df.chunk:
    for y in x.split():
        t.append(y.lower())
Counter(t)

Counter({'the': 271,
         'of': 214,
         '^': 214,
         'and': 181,
         'on': 122,
         'in': 109,
         'retrieved': 105,
         'from': 98,
         'archived': 93,
         'a': 92,
         'original': 88,
         'for': 77,
         'health': 54,
         'to': 53,
         'b': 50,
         'isbn': 46,
         'pmid': 37,
         'clinical': 34,
         'national': 32,
         'research': 30,
         'february': 28,
         'april': 27,
         'p.': 27,
         'january': 27,
         'july': 27,
         'c': 26,
         'd': 25,
         'is': 24,
         'december': 23,
         'march': 23,
         'journal': 22,
         'by': 21,
         '2017.': 21,
         'pp.': 21,
         'disease': 20,
         'control': 20,
         '&': 18,
         '2018.': 18,
         '2015.': 18,
         'september': 17,
         '2023.': 17,
         '2021.': 17,
         'may': 17,
         'et': 16,
         'american': 16,
         'august': 16,
 